# Gated Fusion Experiment: RGB + Depth + Joint Encoders

## Objective
This notebook implements a **multimodal gated fusion model** for posture classification using three pretrained encoders:

- **RGB encoder**
- **Depth encoder**
- **Joint encoder**

The goal is to combine feature representations from all three modalities and train a **gated fusion classification head** on top of them.

---

## Experiment Setup
In this experiment:

- pretrained encoder weights are loaded from saved artifacts
- encoder backbones are used as **feature extractors**
- encoder parameters are initially **frozen**
- a **gated fusion module** is trained to combine modality-specific features
- the final classifier predicts one of the posture classes:
  - **supine**
  - **left**
  - **right**

---

## Why Gated Fusion
Gated fusion allows the model to learn the **relative importance of each modality** for a given sample.

This is useful because:

- **RGB** may become less reliable under blanket occlusion
- **Depth** may retain useful spatial posture information
- **Joint features** may provide structural cues but may also be noisy

The gating mechanism helps the model dynamically weight each modality before final classification.

## Imports and Setup

In [2]:
# =========================
# Core
# =========================
from pathlib import Path

# =========================
# PyTorch
# =========================
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models

In [3]:
# =========================
# Device setup
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Paths and Configuration

In [4]:
# =========================
# Paths
# =========================
PROJECT_ROOT = Path.cwd().resolve().parent

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
EXPERIMENTS_DIR = PROJECT_ROOT / "experiments"

DEPTH_ARTIFACT_DIR = ARTIFACTS_DIR / "depth_encoder_cnn" / "checkpoints"
RGB_ARTIFACT_DIR = ARTIFACTS_DIR / "rgb_encoder_cnn" / "checkpoints"
JOINT_ARTIFACT_DIR = ARTIFACTS_DIR / "joint_xyo" / "checkpoints"

FUSION_CHECKPOINT_DIR = ARTIFACTS_DIR / "checkpoints_fusion_gated"
FUSION_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Depth artifacts:", DEPTH_ARTIFACT_DIR)
print("RGB artifacts:", RGB_ARTIFACT_DIR)
print("Joint artifacts:", JOINT_ARTIFACT_DIR)
print("Fusion checkpoints:", FUSION_CHECKPOINT_DIR)

Project root: C:\Users\arshi\Desktop\AIS\Sem-2\Intelligent Sensing Systems\Practise Module\Project\ApneaSense
Depth artifacts: C:\Users\arshi\Desktop\AIS\Sem-2\Intelligent Sensing Systems\Practise Module\Project\ApneaSense\artifacts\depth_encoder_cnn\checkpoints
RGB artifacts: C:\Users\arshi\Desktop\AIS\Sem-2\Intelligent Sensing Systems\Practise Module\Project\ApneaSense\artifacts\rgb_encoder_cnn\checkpoints
Joint artifacts: C:\Users\arshi\Desktop\AIS\Sem-2\Intelligent Sensing Systems\Practise Module\Project\ApneaSense\artifacts\joint_xyo\checkpoints
Fusion checkpoints: C:\Users\arshi\Desktop\AIS\Sem-2\Intelligent Sensing Systems\Practise Module\Project\ApneaSense\artifacts\checkpoints_fusion_gated


## Experiment Configuration

Define the basic experiment settings for gated fusion, including:

- number of classes
- batch size
- learning rate
- number of epochs
- common feature dimension for fusion

These can be adjusted later if needed.

In [5]:
# =========================
# Experiment config
# =========================
NUM_CLASSES = 3

BATCH_SIZE = 32
LEARNING_RATE = 1e-3
NUM_EPOCHS = 15

COMMON_FEATURE_DIM = 256 # to convert all features from encoders to a common size
DROPOUT = 0.3

CLASS_NAMES = ["supine", "left", "right"]

## Encoder Checkpoint Paths

In [6]:
# =========================
# Encoder checkpoint files
# =========================
DEPTH_CHECKPOINT_PATH = DEPTH_ARTIFACT_DIR / "best_depth_encoder_only.pth"
RGB_CHECKPOINT_PATH = RGB_ARTIFACT_DIR / "best_rgb_encoder.pt"
JOINT_CHECKPOINT_PATH = JOINT_ARTIFACT_DIR / "joint_encoder_xyo_RGB.pth"

print("Depth checkpoint:", DEPTH_CHECKPOINT_PATH)
print("RGB checkpoint:", RGB_CHECKPOINT_PATH)
print("Joint checkpoint:", JOINT_CHECKPOINT_PATH)

Depth checkpoint: C:\Users\arshi\Desktop\AIS\Sem-2\Intelligent Sensing Systems\Practise Module\Project\ApneaSense\artifacts\depth_encoder_cnn\checkpoints\best_depth_encoder_only.pth
RGB checkpoint: C:\Users\arshi\Desktop\AIS\Sem-2\Intelligent Sensing Systems\Practise Module\Project\ApneaSense\artifacts\rgb_encoder_cnn\checkpoints\best_rgb_encoder.pt
Joint checkpoint: C:\Users\arshi\Desktop\AIS\Sem-2\Intelligent Sensing Systems\Practise Module\Project\ApneaSense\artifacts\joint_xyo\checkpoints\joint_encoder_xyo_RGB.pth


## Encoder Model Definitions

Define the encoder architectures needed to load the pretrained weights.

At this stage, we only recreate the model structures required for:

- RGB encoder
- Depth encoder
- Joint encoder

These definitions should match the architectures used during the original encoder training notebooks.

In [7]:
# =========================
# Encoder feature dimensions
# =========================
DEPTH_FEATURE_DIM = 512
RGB_FEATURE_DIM = 128
JOINT_INPUT_DIM = 42
JOINT_FEATURE_DIM = 128

In [8]:
class DepthEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        backbone = models.resnet18(weights=None)

        # change first conv to 1 channel
        backbone.conv1 = nn.Conv2d(
            1, 64, kernel_size=7, stride=2, padding=3, bias=False
        )

        # remove FC
        layers = list(backbone.children())[:-1]

        self.encoder = nn.Sequential(*layers)
        self.flatten = nn.Flatten()

        self.feature_dim = 512

    def forward(self, x):
        x = self.encoder(x)
        x = self.flatten(x)
        return x


class RGBEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        backbone = models.resnet18(weights=None)
        backbone.fc = nn.Identity()

        self.backbone = backbone
        self.projection = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128)
        )
        self.feature_dim = 128

    def forward(self, x):
        x = self.backbone(x)
        x = self.projection(x)
        return x


class JointEncoder(nn.Module):
    def __init__(self, input_dim=42, hidden_dim=128, feature_dim=128, dropout=0.3):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),   # 0
            nn.ReLU(),                          # 1
            nn.Dropout(dropout),               # 2
            nn.Linear(hidden_dim, feature_dim) # 3
        )
        self.feature_dim = feature_dim

    def forward(self, x):
        return self.encoder(x)

## Load Pretrained Encoders

Instantiate the three encoder models and load their encoder-only weights from the saved checkpoints.

After loading:
- move models to the selected device
- set them to evaluation mode
- freeze parameters so that only the gated fusion head is trained in this first experiment stage

In [10]:
# =========================
# Instantiate encoders
# =========================
depth_encoder = DepthEncoder().to(device)
rgb_encoder = RGBEncoder().to(device)
joint_encoder = JointEncoder(
    input_dim=JOINT_INPUT_DIM,
    hidden_dim=128,
    feature_dim=JOINT_FEATURE_DIM,
    dropout=0.3
).to(device)

# =========================
# Load checkpoints
# =========================
depth_ckpt = torch.load(DEPTH_CHECKPOINT_PATH, map_location=device)
rgb_ckpt = torch.load(RGB_CHECKPOINT_PATH, map_location=device)
joint_ckpt = torch.load(JOINT_CHECKPOINT_PATH, map_location=device)

depth_encoder.encoder.load_state_dict(depth_ckpt["encoder_state_dict"])
rgb_encoder.load_state_dict(rgb_ckpt["encoder_state_dict"])
joint_encoder.encoder.load_state_dict(joint_ckpt["encoder_state_dict"])

# =========================
# Freeze encoders
# =========================
for model in [depth_encoder, rgb_encoder, joint_encoder]:
    for param in model.parameters():
        param.requires_grad = False
    model.eval()

print("All encoders loaded and frozen successfully.")

All encoders loaded and frozen successfully.


# Gated Fusion Model